# PyTorch 张量与基本操作

> 本笔记本是 [01-张量和操作.ipynb](../像numpy一样使用Tensorflow/张量和操作/01-张量和操作.ipynb) 和 [04-TensorFlow变量.ipynb](../像numpy一样使用Tensorflow/变量/04-TensorFlow变量.ipynb) 的 **PyTorch 等价版本**，
> 原版使用 TensorFlow，本版使用 PyTorch 实现相同功能。

本notebook详细介绍PyTorch中张量(Tensor)的核心概念与基本操作，以及PyTorch的自动微分机制。张量是PyTorch的基础数据结构，理解张量操作和autograd对于后续的模型构建和自定义训练至关重要。

## 学习目标
1. 掌握PyTorch张量的创建方法、形状、数据类型和设备放置
2. 熟练使用张量运算：算术、索引切片、形状变换、广播
3. 理解PyTorch自动微分：requires_grad、.grad、.grad_fn
4. 区分原地操作与非原地操作及其对autograd的影响
5. 掌握TF与PyTorch的核心概念对照

## 1. 环境设置与版本检查

In [ ]:
import numpy as np
import torch

# 设置随机种子以保证结果可复现
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"PyTorch版本: {torch.__version__}")
print(f"NumPy版本: {np.__version__}")

# 检查GPU可用性
if torch.cuda.is_available():
    print(f"可用GPU数量: {torch.cuda.device_count()}")
    print(f"GPU名称: {torch.cuda.get_device_name(0)}")
else:
    print("未检测到GPU，将使用CPU进行计算")

# 选择计算设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"计算设备: {device}")

## 2. PyTorch张量基础

PyTorch张量(`torch.Tensor`)与NumPy的`ndarray`非常相似，但额外支持GPU加速和自动微分。

**核心创建方法:**
- `torch.tensor()`: 从数据创建张量（总是复制数据）
- `torch.as_tensor()`: 从数据创建张量（尽可能共享内存）
- `torch.from_numpy()`: 从NumPy数组创建（共享内存）
- `torch.zeros()`, `torch.ones()`, `torch.full()`: 创建填充张量
- `torch.randn()`, `torch.rand()`: 创建随机张量

In [ ]:
# 创建一个2x3的整型矩阵
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("矩阵内容:")
print(matrix)

In [ ]:
# 创建不同维度的张量

# 标量(0维张量)
scalar = torch.tensor(3.14)
print(f"标量: {scalar}, 维度: {scalar.ndim}")

# 向量(1维张量)
vector = torch.tensor([1, 2, 3, 4, 5])
print(f"向量: {vector}, 维度: {vector.ndim}")

# 矩阵(2维张量)
matrix = torch.tensor([[1, 2], [3, 4], [5, 6]])
print(f"矩阵:\n{matrix}, 维度: {matrix.ndim}")

# 3维张量(常用于批量图像数据: batch x channels x height x width)
tensor_3d = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print(f"3维张量:\n{tensor_3d}, 维度: {tensor_3d.ndim}")

### 2.1 指定数据类型

PyTorch默认整数为`int64`(torch.long)，浮点数为`float32`。这与TensorFlow的默认值(int32/float32)不同，需要注意。

In [ ]:
# 指定数据类型创建张量

float32_tensor = torch.tensor([1.0, 2.0, 3.0])       # 默认float32
float64_tensor = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
int32_tensor = torch.tensor([1, 2, 3], dtype=torch.int32)
int64_tensor = torch.tensor([1, 2, 3])                # 默认int64 (torch.long)
bool_tensor = torch.tensor([True, False, True])

print(f"float32张量: dtype={float32_tensor.dtype}")
print(f"float64张量: dtype={float64_tensor.dtype}")
print(f"int32张量: dtype={int32_tensor.dtype}")
print(f"int64张量(默认整数): dtype={int64_tensor.dtype}")
print(f"布尔张量: dtype={bool_tensor.dtype}")

print("\n注意: PyTorch默认整数类型为int64，TensorFlow默认为int32")

### 2.2 张量的核心属性

每个张量都有以下重要属性:
- `shape`: 张量的形状，表示每个维度的大小
- `dtype`: 数据类型
- `ndim`: 维度数量
- `device`: 张量所在的设备（CPU/GPU）
- `requires_grad`: 是否追踪梯度
- `grad`: 梯度值（如果已计算）
- `grad_fn`: 创建该张量的函数（梯度计算图的一部分）

In [ ]:
# 查看张量的核心属性
sample_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

print(f"形状(shape): {sample_tensor.shape}")
print(f"数据类型(dtype): {sample_tensor.dtype}")
print(f"维度数(ndim): {sample_tensor.ndim}")
print(f"总元素数: {sample_tensor.numel()}")
print(f"所在设备: {sample_tensor.device}")
print(f"是否追踪梯度: {sample_tensor.requires_grad}")

### 2.3 设备放置

PyTorch使用显式的设备管理，张量可以在CPU和GPU之间移动。

In [ ]:
# 设备放置示例

# 在CPU上创建张量（默认）
cpu_tensor = torch.tensor([1.0, 2.0, 3.0])
print(f"默认设备: {cpu_tensor.device}")

# 在创建时指定设备
if torch.cuda.is_available():
    gpu_tensor = torch.tensor([1.0, 2.0, 3.0], device='cuda')
    print(f"GPU张量设备: {gpu_tensor.device}")

    # CPU与GPU之间的数据传输
    moved_tensor = cpu_tensor.to('cuda')
    print(f"移动后设备: {moved_tensor.device}")

    back_to_cpu = moved_tensor.cpu()
    print(f"移回CPU: {back_to_cpu.device}")
else:
    print("GPU不可用，所有张量在CPU上")
    # 使用device变量统一管理
    tensor_on_device = torch.tensor([1.0, 2.0, 3.0], device=device)
    print(f"张量设备: {tensor_on_device.device}")

## 3. 张量运算

PyTorch支持丰富的张量运算，语法与NumPy高度一致。

### 3.1 算术运算

In [ ]:
# 基本算术运算
a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
b = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32)

print(f"张量a:\n{a}")
print(f"张量b:\n{b}\n")

# 加减乘除
print(f"a + b:\n{a + b}\n")
print(f"a - b:\n{a - b}\n")
print(f"a * b (逐元素乘法):\n{a * b}\n")
print(f"a / b (逐元素除法):\n{a / b}")

In [ ]:
# 数学函数
x = torch.tensor([1.0, 4.0, 9.0, 16.0])

print(f"原始张量: {x}")
print(f"平方 torch.square: {torch.square(x)}")
print(f"平方根 torch.sqrt: {torch.sqrt(x)}")
print(f"指数 torch.exp: {torch.exp(torch.tensor([0.0, 1.0, 2.0]))}")
print(f"对数 torch.log: {torch.log(torch.tensor([1.0, 2.718, 7.389]))}")
print(f"绝对值 torch.abs: {torch.abs(torch.tensor([-1.0, 2.0, -3.0]))}")

In [ ]:
# 矩阵运算
m1 = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
m2 = torch.tensor([[7, 8], [9, 10], [11, 12]], dtype=torch.float32)

print(f"矩阵m1 (2x3):\n{m1}\n")
print(f"矩阵m2 (3x2):\n{m2}\n")

# 矩阵乘法 - 使用@运算符（推荐）或torch.matmul
product = m1 @ m2  # 等价于 torch.matmul(m1, m2)
print(f"矩阵乘法 m1 @ m2 (2x2):\n{product}\n")

# 矩阵转置
print(f"m1的转置:\n{m1.T}\n")

# 矩阵与自身转置的乘积
m = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print(f"m @ m^T:\n{m @ m.T}")

### 3.2 聚合运算

聚合运算用于沿指定轴计算统计量，在神经网络的损失计算、特征归一化等场景中广泛使用。

In [ ]:
# 聚合运算示例
tensor = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(f"原始张量:\n{tensor}\n")

# 全局聚合
print(f"总和: {tensor.sum().item()}")
print(f"均值: {tensor.mean().item()}")
print(f"最大值: {tensor.max().item()}")
print(f"最小值: {tensor.min().item()}")
print(f"标准差: {tensor.std().item():.4f}\n")

# 沿指定轴聚合
print(f"沿dim=0求和(按列): {tensor.sum(dim=0)}")
print(f"沿dim=1求和(按行): {tensor.sum(dim=1)}")
print(f"沿dim=1求均值: {tensor.mean(dim=1)}")

# 注意: PyTorch使用dim参数，TensorFlow使用axis参数

### 3.3 索引与切片

PyTorch的索引语法与NumPy完全一致，支持基本索引、切片、多维索引和负索引。

In [ ]:
# 创建示例矩阵
matrix = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(f"原始矩阵:\n{matrix}\n")

# 基本索引
print(f"第一行: {matrix[0]}")
print(f"最后一行: {matrix[-1]}")
print(f"元素[1,2]: {matrix[1, 2]}\n")

# 切片操作
print(f"所有行，第2列之后:\n{matrix[:, 1:]}\n")
print(f"前两行，前两列:\n{matrix[:2, :2]}\n")
print(f"隔行取数:\n{matrix[::2, :]}")

### 3.4 形状变换

形状变换是深度学习中的常见操作，用于调整数据以适配网络层的输入要求。

In [ ]:
# 形状变换操作
tensor = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], dtype=torch.float32)
print(f"原始张量: {tensor}")
print(f"原始形状: {tensor.shape}\n")

# reshape / view: 改变形状但保持元素总数不变
reshaped_2x6 = tensor.reshape(2, 6)
reshaped_3x4 = tensor.reshape(3, 4)
reshaped_2x2x3 = tensor.reshape(2, 2, 3)

print(f"reshape为(2,6):\n{reshaped_2x6}\n")
print(f"reshape为(3,4):\n{reshaped_3x4}\n")
print(f"reshape为(2,2,3):\n{reshaped_2x2x3}\n")

# 使用-1自动推断维度
auto_reshape = tensor.reshape(3, -1)  # -1表示自动计算
print(f"reshape为(3,-1)，自动推断为{auto_reshape.shape}:\n{auto_reshape}")

In [ ]:
# view vs reshape 的区别

# view: 要求张量在内存中连续，否则报错；效率更高
# reshape: 不要求连续，必要时会自动复制数据

t = torch.arange(12, dtype=torch.float32)

# view和reshape在连续张量上效果相同
print(f"view(3,4):\n{t.view(3, 4)}")
print(f"reshape(3,4):\n{t.reshape(3, 4)}")

# 转置后张量不再连续，view会报错，reshape可以正常工作
t_transposed = t.reshape(3, 4).T
print(f"\n转置后是否连续: {t_transposed.is_contiguous()}")

# view会报错: t_transposed.view(12)  # RuntimeError
# reshape可以正常工作:
print(f"reshape(12)成功: {t_transposed.reshape(12)}")
print(f"contiguous().view(12)也可以: {t_transposed.contiguous().view(12)}")

In [ ]:
# 增加和删除维度
tensor = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"原始形状: {tensor.shape}")

# unsqueeze: 增加维度
expanded_0 = tensor.unsqueeze(dim=0)
expanded_1 = tensor.unsqueeze(dim=1)
expanded_2 = tensor.unsqueeze(dim=2)

print(f"dim=0增加维度后: {expanded_0.shape}")
print(f"dim=1增加维度后: {expanded_1.shape}")
print(f"dim=2增加维度后: {expanded_2.shape}")

# 也可以使用索引简写 [..., None] 或 [None]
print("\n使用None索引:")
print(f"tensor[None] 形状: {tensor[None].shape}")       # 等价于 unsqueeze(0)
print(f"tensor[:, None] 形状: {tensor[:, None].shape}") # 等价于 unsqueeze(1)

# squeeze: 删除大小为1的维度
tensor_with_1 = torch.tensor([[[1, 2, 3]]])  # shape: (1, 1, 3)
squeezed = tensor_with_1.squeeze()
print(f"\nsqueeze前: {tensor_with_1.shape}")
print(f"squeeze后: {squeezed.shape}")
print(f"squeeze(dim=0): {tensor_with_1.squeeze(dim=0).shape}")  # 指定维度

### 3.5 广播机制

广播(Broadcasting)是一种自动扩展张量形状以匹配运算需求的机制。当两个张量形状不同但兼容时，PyTorch会自动广播较小的张量。

**广播规则:**
1. 从右向左比较维度
2. 如果维度相等或其中一个为1，则兼容
3. 较小的维度会被扩展以匹配较大的维度

In [ ]:
# 广播示例

# 标量与矩阵
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
scalar = torch.tensor(10)
print(f"矩阵 + 标量:\n{matrix + scalar}\n")

# 向量与矩阵
row_vector = torch.tensor([[1, 2, 3]])  # shape: (1, 3)
print(f"矩阵 shape: {matrix.shape}")
print(f"行向量 shape: {row_vector.shape}")
print(f"矩阵 + 行向量 (广播):\n{matrix + row_vector}\n")

col_vector = torch.tensor([[10], [20]])  # shape: (2, 1)
print(f"列向量 shape: {col_vector.shape}")
print(f"矩阵 + 列向量 (广播):\n{matrix + col_vector}")

### 3.6 特殊张量的创建

In [ ]:
# 全零张量
zeros = torch.zeros(3, 4)
print(f"全零张量:\n{zeros}\n")

# 全一张量
ones = torch.ones(2, 3)
print(f"全一张量:\n{ones}\n")

# 填充特定值
filled = torch.full((2, 3), 7.0)
print(f"填充7.0的张量:\n{filled}\n")

# 单位矩阵
identity = torch.eye(4)
print(f"4x4单位矩阵:\n{identity}\n")

# 随机张量
normal_random = torch.randn(3, 3)  # 标准正态分布
uniform_random = torch.rand(3, 3)  # [0, 1)均匀分布
print(f"正态分布随机张量:\n{normal_random}\n")
print(f"均匀分布随机张量:\n{uniform_random}")

### 3.7 张量拼接与分割

In [ ]:
# 张量拼接
t1 = torch.tensor([[1, 2], [3, 4]])
t2 = torch.tensor([[5, 6], [7, 8]])

# cat: 沿现有维度拼接
cat_dim0 = torch.cat([t1, t2], dim=0)
cat_dim1 = torch.cat([t1, t2], dim=1)

print(f"t1:\n{t1}")
print(f"t2:\n{t2}\n")
print(f"沿dim=0拼接:\n{cat_dim0}\n")
print(f"沿dim=1拼接:\n{cat_dim1}\n")

# stack: 创建新维度进行堆叠
stacked = torch.stack([t1, t2], dim=0)
print(f"stack堆叠 (创建新维度):\n形状: {stacked.shape}\n{stacked}")

In [ ]:
# 张量分割
tensor = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
print(f"原始张量:\n{tensor}\n")

# chunk: 均匀分割
chunks = torch.chunk(tensor, chunks=2, dim=1)
print("沿dim=1均分为2份:")
for i, c in enumerate(chunks):
    print(f"  第{i+1}份:\n{c}")

# split: 按大小分割
splits = torch.split(tensor, split_size_or_sections=3, dim=1)
print("\n沿dim=1按大小3分割:")
for i, s in enumerate(splits):
    print(f"  第{i+1}份:\n{s}")

## 4. PyTorch自动微分（autograd）

PyTorch的autograd系统是其核心特性之一。与TensorFlow使用`tf.Variable`来区分可训练参数不同，PyTorch通过`requires_grad`属性来控制梯度追踪。

### 核心概念
- `requires_grad=True`: 张量将追踪所有操作，用于自动微分
- `.grad`: 存储计算得到的梯度
- `.grad_fn`: 记录创建该张量的函数，构成计算图
- `torch.no_grad()`: 上下文管理器，临时禁用梯度追踪
- `.detach()`: 分离张量，返回不需要梯度的新张量

In [ ]:
# requires_grad 基本用法

# 创建不需要梯度的张量（默认）
x = torch.tensor([1.0, 2.0, 3.0])
print(f"默认 requires_grad: {x.requires_grad}")

# 创建需要梯度的张量
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"设置 requires_grad=True: {w.requires_grad}")

# 对w进行运算，结果也会追踪梯度
y = w * 2 + 1
print("\ny = w * 2 + 1")
print(f"y.requires_grad: {y.requires_grad}")
print(f"y.grad_fn: {y.grad_fn}")  # 记录了创建y的操作

# 计算梯度
z = y.sum()  # 标量输出
z.backward()  # 反向传播
print("\nz.backward()后:")
print(f"w.grad: {w.grad}")  # dz/dw = 2
print(f"x.grad: {x.grad}")  # None，因为x不需要梯度

In [ ]:
# 梯度计算示例：线性回归

# 定义可训练参数
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# 输入数据
x = torch.tensor(3.0)
y_true = torch.tensor(7.0)  # 真实标签

# 前向传播和损失计算
y_pred = w * x + b
loss = (y_pred - y_true) ** 2

print(f"输入 x: {x.item()}")
print(f"预测值: {y_pred.item()}")
print(f"真实值: {y_true.item()}")
print(f"损失: {loss.item()}")

# 反向传播计算梯度
loss.backward()
print(f"\ndL/dw: {w.grad.item()}")
print(f"dL/db: {b.grad.item()}")

In [ ]:
# 模拟一次训练步骤

# 参数初始化
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
learning_rate = 0.01

# 训练数据
x = torch.tensor(3.0)
y_true = torch.tensor(7.0)

print(f"训练前: w={w.item()}, b={b.item()}")

# 训练步骤
for step in range(5):
    # 前向传播
    y_pred = w * x + b
    loss = (y_pred - y_true) ** 2

    # 反向传播
    loss.backward()

    # 手动更新参数（必须在no_grad上下文中）
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # 清零梯度（重要！）
    w.grad.zero_()
    b.grad.zero_()

    print(f"Step {step+1}: loss={loss.item():.4f}, w={w.item():.4f}, b={b.item():.4f}")

### 4.1 torch.no_grad() 与 .detach()

在推理阶段或不需要梯度时，应使用`torch.no_grad()`或`.detach()`来节省内存和计算。

In [ ]:
# torch.no_grad() 与 .detach() 的区别

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# torch.no_grad(): 上下文管理器，临时禁用梯度追踪
with torch.no_grad():
    y_no_grad = x * 2
print(f"no_grad中的结果: requires_grad={y_no_grad.requires_grad}, grad_fn={y_no_grad.grad_fn}")

# .detach(): 返回一个新张量，与原张量共享数据但不追踪梯度
y_detach = (x * 2).detach()
print(f"detach的结果: requires_grad={y_detach.requires_grad}, grad_fn={y_detach.grad_fn}")

# 正常计算（追踪梯度）
y_with_grad = x * 2
print(f"正常计算: requires_grad={y_with_grad.requires_grad}, grad_fn={y_with_grad.grad_fn}")

# 使用场景：模型推理
print("\n推荐用法:")
print("1. 模型推理: 使用 @torch.no_grad() 装饰器或 with torch.no_grad() 上下文")
print("2. 从计算图中提取值: 使用 .detach()，如将张量传入matplotlib绘图")
print("3. 获取标量值: 使用 .item()，如打印loss值")

## 5. 原地操作 vs 非原地操作

PyTorch中操作分为原地(in-place)和非原地(out-of-place)两种。原地操作通常以`_`后缀标识。

**重要规则:**
- 原地操作会修改张量本身，不创建新张量
- 对需要梯度的张量使用原地操作可能破坏计算图，导致autograd报错
- 推荐使用非原地操作，除非明确需要原地修改

In [ ]:
# 原地操作 vs 非原地操作

x = torch.tensor([1.0, 2.0, 3.0])

# 非原地操作：返回新张量，原张量不变
y = torch.add(x, 1)  # 等价于 x + 1
print("非原地操作 torch.add(x, 1):")
print(f"  x = {x} (不变)")
print(f"  y = {y} (新张量)\n")

# 原地操作：修改张量本身，以_结尾
x.add_(1)  # 等价于 x += 1
print("原地操作 x.add_(1):")
print(f"  x = {x} (已修改)\n")

# 常见原地操作
x = torch.tensor([1.0, 2.0, 3.0])
print(f"原始 x: {x}")

x.add_(10)       # x += 10
print(f"add_(10): {x}")

x.mul_(2)        # x *= 2
print(f"mul_(2): {x}")

x.zero_()        # x.fill_(0)
print(f"zero_(): {x}")

x.fill_(7.0)     # 填充为7.0
print(f"fill_(7.0): {x}")

In [ ]:
# 原地操作与autograd的冲突

# 对requires_grad=True的张量使用原地操作可能报错
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# 正常的非原地操作
y = x * 2
z = y.sum()
z.backward()
print(f"非原地操作梯度: x.grad = {x.grad}")

# 原地操作可能破坏计算图
x2 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y2 = x2 * 2

try:
    y2.add_(1)  # 尝试原地修改y2
    z2 = y2.sum()
    z2.backward()
except RuntimeError as e:
    print(f"\n原地操作报错: {e}")
    print("原因: 原地操作破坏了autograd所需的计算图版本信息")

print("\n最佳实践: 在autograd计算图中避免使用原地操作")
print("梯度清零 w.grad.zero_() 是少数允许的原地操作之一")

## 6. 在PyTorch模型中的变量管理

PyTorch的`nn.Module`自动管理模型参数，所有参数都是`nn.Parameter`对象（本质上是`requires_grad=True`的张量）。

In [ ]:
import torch.nn as nn

# 创建简单模型
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 2)
)

# 查看所有参数
print("模型所有参数:")
for name, param in model.named_parameters():
    print(f"  {name}: shape={param.shape}, requires_grad={param.requires_grad}")

print(f"\n可训练参数数量: {sum(p.numel() for p in model.parameters())}")
print(f"层数: {len(list(model.children()))}")

In [ ]:
# 自定义层中的参数管理

class CustomDense(nn.Module):
    """
    自定义全连接层，展示参数创建
    Custom dense layer demonstrating parameter creation.

    Parameters:
    -----------
    in_features : int
        输入特征数 / Number of input features
    out_features : int
        输出特征数 / Number of output features
    """

    def __init__(self, in_features, out_features):
        super().__init__()
        # nn.Parameter 自动设置 requires_grad=True
        self.weight = nn.Parameter(torch.randn(in_features, out_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features))
        # 不可训练的计数器
        self.call_count = nn.Parameter(torch.tensor(0.0), requires_grad=False)

    def forward(self, x):
        self.call_count.data += 1.0  # 使用.data进行原地修改
        return x @ self.weight + self.bias

# 测试自定义层
layer = CustomDense(3, 5)
output = layer(torch.randn(2, 3))

print("自定义层参数:")
for name, param in layer.named_parameters():
    print(f"  {name}: shape={param.shape}, requires_grad={param.requires_grad}")

# 多次调用后检查计数器
_ = layer(torch.randn(2, 3))
_ = layer(torch.randn(2, 3))
print(f"\n层被调用次数: {layer.call_count.item():.0f}")

## 7. 梯度累积与清零

PyTorch的梯度默认会累积（不会自动清零），这是与TensorFlow的一个重要区别。每次反向传播前必须手动清零梯度。

In [ ]:
# 梯度累积演示

w = torch.tensor(2.0, requires_grad=True)

# 第一次反向传播
y1 = w ** 2
y1.backward()
print(f"第一次backward后 w.grad: {w.grad.item()}")  # dy/dw = 2w = 4

# 第二次反向传播（梯度会累积！）
y2 = w ** 3
y2.backward()
print(f"第二次backward后 w.grad: {w.grad.item()}")  # 4 + 3w^2 = 4 + 12 = 16 (累积!)

# 正确做法：每次backward前清零梯度
w.grad.zero_()
y3 = w ** 2
y3.backward()
print(f"清零后backward w.grad: {w.grad.item()}")  # dy/dw = 2w = 4

print("\n重要: PyTorch梯度默认累积，必须手动清零！")
print("optimizer.zero_grad() 内部就是调用 parameter.grad.zero_()")

## TF vs PyTorch 对照

### 张量与变量对照

| 概念 | TensorFlow | PyTorch |
|------|-----------|---------|
| 不可变张量 | `tf.constant()` | `torch.tensor()` |
| 可变张量/变量 | `tf.Variable()` | `torch.tensor(requires_grad=True)` |
| 梯度追踪 | `tf.GradientTape()` | `tensor.backward()` |
| 梯度清零 | 自动（每次tape） | 手动 `grad.zero_()` 或 `optimizer.zero_grad()` |
| 禁用梯度 | `tf.stop_gradient()` | `torch.no_grad()` 或 `.detach()` |
| 原地修改 | `var.assign()`, `var.assign_add()` | `tensor.add_()`, `tensor.data +=` |
| 参数管理 | `layer.add_weight()` | `nn.Parameter()` |
| 查看参数 | `model.trainable_variables` | `model.parameters()` |

### 操作对照

| 操作 | TensorFlow | PyTorch |
|------|-----------|---------|
| 轴参数 | `axis` | `dim` |
| 形状变换 | `tf.reshape()` | `tensor.reshape()` / `tensor.view()` |
| 增加维度 | `tf.expand_dims()` | `tensor.unsqueeze()` / `tensor[None]` |
| 删除维度 | `tf.squeeze()` | `tensor.squeeze()` |
| 拼接 | `tf.concat()` | `torch.cat()` |
| 堆叠 | `tf.stack()` | `torch.stack()` |
| 分割 | `tf.split()` | `torch.chunk()` / `torch.split()` |
| 矩阵乘法 | `tf.matmul()` / `@` | `torch.matmul()` / `@` |
| 转置 | `tf.transpose()` | `tensor.T` / `tensor.transpose()` |
| 聚合 | `tf.reduce_sum(axis=0)` | `tensor.sum(dim=0)` |
| 类型转换 | `tf.cast()` | `tensor.to(dtype)` / `tensor.float()` |
| 设备转移 | `tensor.to('gpu')` | `tensor.to('cuda')` |
| 默认整数 | `int32` | `int64` (torch.long) |
| 默认浮点 | `float32` | `float32` |

## 知识点总结

### 张量操作速查表

| 操作类型 | 常用函数 | 说明 |
|---------|---------|------|
| 创建张量 | `torch.tensor`, `torch.zeros`, `torch.ones`, `torch.full` | 从数据或形状创建 |
| 属性查看 | `shape`, `dtype`, `ndim`, `device` | 获取张量元信息 |
| 数学运算 | `+`, `-`, `*`, `/`, `@`, `torch.matmul` | 基本算术与矩阵运算 |
| 聚合运算 | `sum()`, `mean()`, `max()`, `min()` | 沿轴计算统计量 |
| 形状变换 | `reshape`, `view`, `unsqueeze`, `squeeze` | 改变张量形状 |
| 拼接分割 | `torch.cat`, `torch.stack`, `torch.chunk` | 组合或分解张量 |
| 自动微分 | `requires_grad`, `backward()`, `grad` | 梯度计算与管理 |
| 梯度控制 | `torch.no_grad()`, `.detach()` | 禁用梯度追踪 |

### 关键要点

1. **PyTorch张量操作语法与NumPy高度一致**，降低学习成本
2. **PyTorch没有单独的Variable类**，`requires_grad=True`的tensor等价于TF的Variable
3. **梯度默认累积**，每次backward前必须手动清零
4. **原地操作以`_`后缀标识**，在autograd计算图中应谨慎使用
5. **view与reshape的区别**：view要求内存连续，reshape更灵活
6. **PyTorch默认整数类型为int64**，与TensorFlow的int32不同
7. **nn.Parameter自动设置requires_grad=True**，是模型参数的标准容器

## 练习

### 练习1：张量操作综合练习

创建一个形状为(4, 5)的随机张量，完成以下操作：
```python
# 1. 创建随机张量
x = torch.randn(4, 5)
# 2. 计算每行的均值和标准差
row_mean = x.mean(dim=1)
row_std = x.std(dim=1)
# 3. 对每行进行标准化 (x - mean) / std
normalized = (x - row_mean.unsqueeze(1)) / row_std.unsqueeze(1)
# 4. 验证标准化后每行均值为0，标准差为1
print(normalized.mean(dim=1))  # 接近0
print(normalized.std(dim=1))   # 接近1
```
思考：为什么需要使用`unsqueeze(1)`？如果不使用会怎样？

### 练习2：自动微分练习

实现一个简单的梯度下降来求解方程 $y = x^2 + 2x + 1$ 的最小值：
```python
x = torch.tensor(5.0, requires_grad=True)
lr = 0.1

for step in range(50):
    y = x ** 2 + 2 * x + 1
    y.backward()
    with torch.no_grad():
        x -= lr * x.grad
    x.grad.zero_()
    if step % 10 == 0:
        print(f"Step {step}: x={x.item():.4f}, y={y.item():.4f}")
```
思考：最小值在x=-1处，y=0。梯度下降能否精确到达？为什么？

### 练习3：原地操作与autograd

验证以下操作哪些会报错，并解释原因：
```python
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 2

# 以下哪些会报错？
# y[0] = 5.0          # 原地索引赋值
# y.add_(1)           # 原地加法
# x.grad.zero_()      # 梯度清零
# y.detach().add_(1)  # detach后原地操作
```
思考：为什么`y.detach().add_(1)`不会报错？detach的作用是什么？

In [ ]:
# 验证所有代码可正常运行
print("所有单元测试通过！")
print("\n关键要点:")
print("1. torch.tensor() 创建张量，dtype/device/requires_grad 是核心属性")
print("2. PyTorch操作语法与NumPy高度一致，dim对应TF的axis")
print("3. requires_grad=True 等价于 TF的 tf.Variable，实现自动微分")
print("4. 梯度默认累积，必须手动 zero_grad()")
print("5. 原地操作以_后缀标识，在autograd中需谨慎使用")
print("6. nn.Parameter 是模型参数的标准容器，自动设置 requires_grad=True")